    Instructions:
    1. Run task_0a.py to generate the vector database if not already generated.
    2. Press Run All (or restart kernel and run all cells).
    3. You will be prompted to provide input values.


    – Task 3 (LS1): Implement a program which (a) given one of the feature models, 
    (b) a user specified value of k, (c) one of the four dimensionality reduction 
    techniques (SVD, NNMF, LDA, k-means) chosen by the user, reports the top-k 
    latent semantics extracted under the selected feature space.

    – Store the latent semantics in a properly named output file

    – List imageID-weight pairs, ordered in decreasing order of weights

In [6]:
FEATURE_SPACE = input("Provide a feature space [color, hog, avgpool, layer3, fc, resnet_output].")

DIM_REDUCTION = input("Provide a dimensionality reduction technique [svd, nnmf, lda, kmeans].")

K = int(input("Enter K, the top K latent semantics to extract for the selected feature space."))

In [7]:
from utils.database_utils import retrieve
feature_vectors = retrieve(f'{FEATURE_SPACE}.pt')

print("Generating top-", K, " latent semantics under ", FEATURE_SPACE, " feature space using: ", DIM_REDUCTION)

Generating top- 5  latent semantics under  color  feature space using:  nnmf


In [8]:
if DIM_REDUCTION == "svd":
    from feature_reducers.svd import SVDReducer
    reducer = SVDReducer


elif DIM_REDUCTION == "nnmf":
    from feature_reducers.nnmf import NNMFReducer
    reducer = NNMFReducer

elif DIM_REDUCTION == "lda":
    from feature_reducers.lda import LDAReducer
    reducer = LDAReducer

else:
    # kmeans.
    from feature_reducers.kmeans import KMeansReducer
    reducer = KMeansReducer

reducer = reducer(feature_vectors, K)

similarity_matrix = reducer.get_similarity_matrix(feature_vectors)

latent_semantics = reducer.reduce_features(feature_vectors)
print("Top K latent semantics: ")
print(latent_semantics)
print("Shape: ", latent_semantics.shape)

Top K latent semantics: 
[[0.33981897 0.7832111  1.16432383 1.04293075 1.13822838]
 [1.42827774 0.82033476 1.63247842 0.29264402 0.57715107]
 [0.18198112 0.52942475 0.81289882 1.01712473 1.45534491]
 ...
 [1.48624606 1.35103918 0.51657088 0.50881215 0.57967303]
 [1.74471521 0.50587412 1.65058493 0.52941479 0.45000305]
 [2.31280827 0.47907605 1.51379054 0.49373077 0.25697983]]
Shape:  (4339, 5)


In [9]:
# Store the latent semantics in a properly named file.
# We opt to store just the reducer, as we anyway can generate the latent space quickly
# by loading the feature space and passing it to the reducer, eg:
#
# unpicked_reducer = retrieve(f'color_svd_reducer.pt')
# feature_vectors = retrieve(f'color.pt')
#
# unpickled_reducer.reduce_features(feature_vectors)

from utils.database_utils import store

store(reducer, f'LS1_{FEATURE_SPACE}_{DIM_REDUCTION}_reducer.pt')


 Saving:  LS1_color_nnmf_reducer.pt 



In [10]:
# List imageID-weight pairs, ordered in decreasing order of weights

# We are to showcase which images contribute more to each latent feature.
# This is taking the object-feature factor matrix, and sorting by each
# latent feature's weight.

image_weight_tuples = list(zip(feature_vectors.keys(), similarity_matrix))

print("Image ID - weight pairs sorted in descending order of weights for each latent feature:")

for i in range(K):
    print("\n\nLatent feature: ", i + 1)
    for IMG_ID, weight in sorted(image_weight_tuples, key=lambda x : x[1][i], reverse=True):
        print("(ID: ", IMG_ID, ", Weight: ", weight[i], end="),\t")

Image ID - weight pairs sorted in descending order of weights for each latent feature:


Latent feature:  1
(ID:  1652 , Weight:  3.6223851038144166),	(ID:  1798 , Weight:  3.620608840652698),	(ID:  1802 , Weight:  3.6155690709494874),	(ID:  1806 , Weight:  3.6006044515183437),	(ID:  1642 , Weight:  3.5994422489629807),	(ID:  1790 , Weight:  3.5806306132564347),	(ID:  1784 , Weight:  3.545116742402127),	(ID:  1786 , Weight:  3.5427044145671935),	(ID:  1788 , Weight:  3.482269186365247),	(ID:  1470 , Weight:  3.4617021045953043),	(ID:  1792 , Weight:  3.4223872891106226),	(ID:  1808 , Weight:  3.4022826659979177),	(ID:  1800 , Weight:  3.3915879716275534),	(ID:  1814 , Weight:  3.3671129610391746),	(ID:  1632 , Weight:  3.342290702565846),	(ID:  1796 , Weight:  3.3290420728985426),	(ID:  1852 , Weight:  3.3277667339611012),	(ID:  6492 , Weight:  3.3170272244580516),	(ID:  1832 , Weight:  3.3133850037006853),	(ID:  1794 , Weight:  3.3120515303555114),	(ID:  1804 , Weight:  3.306663578927